<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
Supplementary code for the <a href="https://mng.bz/lZ5B">Build a Reasoning Model (From Scratch)</a> book by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>Code repository: <a href="https://github.com/rasbt/reasoning-from-scratch">https://github.com/rasbt/reasoning-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="https://mng.bz/lZ5B"><img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

# Bölüm 6: Pekiştirmeli Öğrenmeyle Akıl Yürütme Modelleri Eğitmek

Bu not defterinde kullanılan paketler:

In [1]:
from importlib.metadata import version

used_libraries = [
    "reasoning_from_scratch",
    "torch",
    "tokenizers"  # Used by reasoning_from_scratch
]

for lib in used_libraries:
    print(f"{lib} version: {version(lib)}")

reasoning_from_scratch version: 0.1.16
torch version: 2.10.0
tokenizers version: 0.21.4


<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F01_raschka.webp" width=600>

&nbsp;
## 6.1 LLM'ler için pekiştirmeli öğrenmeye giriş

- Çıkarım anında ölçekleme, üretilen yanıt başına daha fazla hesaplama kullanarak akıl yürütmeyi iyileştirir
- Eğitim anında ölçekleme ise eğitim sırasında ek hesaplama kullanarak akıl yürütmeyi iyileştirir; bu bölümün odağı budur

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F02_raschka.webp" width=600>

- Çıkarım anında ölçekleme ile eğitim anında ölçekleme birlikte de kullanılabilir (/kullanılmalıdır); örneğin, RL tabanlı akıl yürütme eğitiminden sonra çıkarım anı teknikleri uygulanarak
- Pratikte LLM'ler için RL, önceden eğitilmiş bir modelin üzerine ya da talimat ince ayarının ardından bir eğitim sonrası aşama olarak uygulanır

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F03_raschka.webp" width=600>

- Ön eğitim, sonraki token tahminiyle genel bilgi oluşturur; RL ise yanıt doğruluğu ya da tercihler gibi dizi düzeyindeki hedefleri optimize ederek model davranışını iyileştirir
- LLM'ler için RL, akıl yürütme eğitimini ve tercih ayarını kapsar; ancak DeepSeek-R1'in gösterdiği gibi akıl yürütme odaklı RL doğrudan önceden eğitilmiş bir temel modele de uygulanabilir
- Akıl yürütmeyi doğrudan temel model üzerinde eğitmek daha zayıf ama yine de yetenekli bir model üretir (ancak akıl yürütme aşamasının neye katkı sağladığını anlamak için daha basit bir ortam sunar)

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F04_raschka.webp" width=600>

&nbsp;
### 6.1.1 İnsan geri bildirimiyle özgün pekiştirmeli öğrenme hattı (RLHF)

- RLHF, 2022'de InstructGPT çalışmasıyla tanıtıldı ve LLM'leri eğitmek için insan tercih etiketleri kullanır (bu, GPT-3'ü özgün ChatGPT'ye dönüştürmede kilit bir adımdı)
- Sonraki token tahminini optimize eden ön eğitim ve denetimli ince ayarın aksine RLHF, modelleri model yanıtlarına ilişkin insan tercih etiketlerine göre optimize eder

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F05_raschka.webp" width=600>

&nbsp;
### 6.1.2 İnsan geri bildiriminden doğrulanabilir ödüllere (RLVR)

- RLHF, çoğu zaman büyük ve maliyetli bir LLM olan ayrı bir ödül modelinin eğitilmesini gerektirir
- RLVR ise öğrenilen ödül modelini otomatik olarak doğrulanabilir, belirlenimci ödüllerle değiştirir

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F06_raschka.webp" width=600>

- RLVR'nin yaygınlaşması büyük ölçüde, insan tercih verisine ya da öğrenilmiş bir ödül modeline dayanmadan güçlü akıl yürütme başarımı sergileyen DeepSeek-R1'in 2025'teki başarısıyla oldu
- DeepSeek-R1, akıl yürütme davranışını matematik problemleri için doğruluk kontrolleri ve programlama görevleri için kod derleme veya çalıştırma gibi otomatik olarak doğrulanabilir ödüllerle eğitti
- Bu kitap matematik tabanlı doğrulamaya odaklansa da, altta yatan fikir kod doğrulamasına benzer: ödüller ikili başarı sinyalleri kullanılarak otomatik olarak hesaplanır

&nbsp;
## 6.2 GRPO kullanarak doğrulanabilir ödüllerle pekiştirmeli öğrenme turu

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F07_raschka.webp" width=600>

- Genel resmi tanıttıktan ve RL'nin LLM geliştirme döngüsüne nasıl oturduğunu gördükten sonra, bir akıl yürütme modeli eğitmek için RLVR uygulayacağız (DeepSeek-R1-Zero'ya benzer, ancak çok daha küçük ölçekte; karşılaştırılabilir bir çalıştırma yüz binlerce dolarlık GPU maliyeti anlamına gelirdi)
- LLM'ler için RL, eğitmek istediğimiz LLM'i (RL bağlamlarında "politika" denir) güncellemekte kullanılan, politika gradyanı adı verilen bir algoritma kullanır
- RLHF için popüler bir politika gradyanı algoritması yakınsal politika optimizasyonudur (PPO); aynı algoritmayı RLVR'de de kullanabilirdik
- Ancak DeepSeek ekibi, DeepSeek-R1 akıl yürütme modellerini eğitirken daha basit bir algoritma kullandı: grup göreli politika optimizasyonu (GRPO) (ilk kez DeepSeekMath'te kullanıldı)
- GRPO kaynak açısından daha dosttur; çünkü PPO'da değer fonksiyonunu hesaplayan başka bir LLM vardır; GRPO'da buna gerek yoktur, zira öğrenme sinyalini örneklenen bir yanıt grubu içindeki göreli karşılaştırmalardan türetir
- İlgilenen okurlar PPO ile GRPO arasında daha ayrıntılı bir yan yana karşılaştırmayı [The State of Reinforcement Learning for LLM Reasoning](https://magazine.sebastianraschka.com/p/the-state-of-llm-reasoning-model-training) yazımda bulabilir
- Bu bölümde RLVR'yi GRPO kullanarak uyguluyoruz
- Ayrıca bir sonraki bölüm, eğitim kararlılığını ve elde edilen modelleme başarımını artırmak için GRPO'ya ek iyileştirmeler tanıtıyor

### 6.2.1 Bir aşçı benzetmesiyle GRPO'ya üst düzey sezgi

- GRPO ilk bakışta karmaşık görünebildiği için, bu bölüme terminolojiyi tanıtmak ve biraz sezgi kazandırmak amacıyla "aşçı ve yemek pişirme" benzetmesiyle genel bir bakışla başlamak istedim

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F08_raschka.webp" width=600>

- LLM'ler için RL bağlamlarının çoğunda rollout ve tamamlama terimleri birbirinin yerine kullanılır

### 6.2.2 Üst düzey GRPO yordamı

- Sonraki bölümlerde GRPO'yu uygulamaya yönelik teknik yol haritası:

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F09_raschka.webp" width=600>

&nbsp;
## 6.3 Önceden eğitilmiş bir model yüklemek

- Bu bölümdeki kod önceki bölümlerdekiyle aynıdır

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F10_raschka.webp" width=600>

In [2]:
import torch

from reasoning_from_scratch.ch02 import get_device
from reasoning_from_scratch.ch03 import (
     load_model_and_tokenizer
)

device = get_device()
device = torch.device("cpu")

model, tokenizer = load_model_and_tokenizer(
    which_model="base",
    device=device,
    use_compile=False
)

Using Apple Silicon GPU (MPS)
✓ qwen3/qwen3-0.6B-base.pth already up-to-date


In [3]:
from reasoning_from_scratch.ch03 import render_prompt
from reasoning_from_scratch.ch04 import (
    generate_text_stream_concat_flex,
    generate_text_top_p_stream_cache
)

raw_prompt = (
    "Half the value of $3x-9$ is $x+37$. "
    "What is the value of $x$?"
)
prompt = render_prompt(raw_prompt)

torch.manual_seed(0)
response = generate_text_stream_concat_flex(
    model, tokenizer, prompt, device,
    max_new_tokens=2048, verbose=True,
    generate_func=generate_text_top_p_stream_cache,
    temperature=0.9,
    top_p=0.9
)

 \boxed{58}

&nbsp;
## 6.4 Bir MATH eğitim alt kümesini yüklemek

- Özgün MATH veri kümesinden türetilen ve önceki bölümlerde model değerlendirmesi için kullanılan MATH-500 örneklerini açıkça dışarıda bırakan, örtüşmeyen bir eğitim alt kümesi kullanıyoruz (veri kümesinin nasıl hazırlandığına dair daha fazla bilgi için bkz. https://github.com/rasbt/math_full_minus_math500)

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F11_raschka.webp" width=600>

- Aşağıdaki `load_math_train` fonksiyonu, farklı bir dosya yolu belirtmemiz dışında 3. bölümdeki [load_math500_test](https://github.com/rasbt/reasoning-from-scratch/blob/main/reasoning_from_scratch/ch03.py#L422) fonksiyonuna benzer

In [4]:
import json
import requests
from pathlib import Path

def load_math_train(local_path="math_train.json", save_copy=True):
    local_path = Path(local_path)

    url = (
        "https://raw.githubusercontent.com/rasbt/"
        "math_full_minus_math500/refs/heads/main/"
        "math_full_minus_math500.json"
    )
    backup_url = (
        "https://f001.backblazeb2.com/file/reasoning-from-scratch/"
        "MATH/math_full_minus_math500.json"
    )

    if local_path.exists():
        with local_path.open("r", encoding="utf-8") as f:
            data = json.load(f)
    else:
        try:
            r = requests.get(url, timeout=30)
            r.raise_for_status()
        except requests.RequestException:
            print("Using backup URL.")
            r = requests.get(backup_url, timeout=30)
            r.raise_for_status()

        data = r.json()

        if save_copy:
            with local_path.open("w", encoding="utf-8") as f:
                json.dump(data, f, indent=2)

    return data

In [5]:
math_train = load_math_train()

print("Dataset size:", len(math_train))

Dataset size: 12000


In [6]:
from pprint import pprint

pprint(math_train[4])

{'answer': '6',
 'level': 'Level 3',
 'problem': 'Sam is hired for a 20-day period. On days that he works, he earns '
            '$\\$$60. For each day that he does not work, $\\$$30 is '
            'subtracted from his earnings. At the end of the 20-day period, he '
            'received $\\$$660. How many days did he not work?',
 'solution': 'Call $x$ the number of days Sam works and $y$ the number of days '
             'he does not. We can set up the following system of equations to '
             'represent the given information: \\begin{align*}\n'
             'x+y &= 20 \\\\\n'
             '60x - 30y &= 660 \\\\\n'
             '\\end{align*} The first equation represents the total number of '
             'days Sam works, and the second equation represents his total '
             'profit. Solving for $x$ in the first equation yields $x = 20 - '
             'y$. Substituting into the second equation gives $60(20-y) - 30y '
             '= 660$. Canceling a factor of $10$ an

- Yalnızca `"answer"` ve `"problem"` alanlarına ihtiyacımız olduğunu unutmayın
- Kuramsal olarak `"solution"` alanını kullanmak cazip gelebilir; ancak burada modelin çözümleri serbestçe keşfetmesini istiyoruz (belirli bir çözümü ve biçimi öğrenmesini değil)

&nbsp;
## 6.5 Rollout örneklemek

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F12_raschka.webp" width=600>

- Rollout, üretilen yanıt için kullanılan RL jargonudur

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F13_raschka.webp" width=400>

- Bir çizge oluşturup bunun üzerinden geri yayılım yapmak istemediğimiz için `@torch.no_grad` gerekiyor; ancak `@torch.inference_mode` işe yaramıyor (fazlasını yapıyor) ve şu hataya yol açıyor

> RuntimeError: Inference tensors cannot be saved for backward. Please do not use Tensors created in inference mode in computation tracked by autograd. To work around this, you can make a clone to get a normal tensor and use it in autograd, or use `torch.no_grad()` instead of `torch.inference_mode()`.

In [7]:
from reasoning_from_scratch.qwen3 import KVCache
from reasoning_from_scratch.ch04 import top_p_filter


@torch.no_grad()
def sample_response(
    model,
    tokenizer,
    prompt,
    device,
    max_new_tokens=512,
    temperature=0.8,
    top_p=0.9,
):
    input_ids = torch.tensor(
        tokenizer.encode(prompt),
        device=device
        )

    cache = KVCache(n_layers=model.cfg["n_layers"])
    model.reset_kv_cache()
    logits = model(input_ids.unsqueeze(0), cache=cache)[:, -1]

    generated = []
    for _ in range(max_new_tokens):
        if temperature and temperature != 1.0:
            logits = logits / temperature

        probas = torch.softmax(logits, dim=-1)
        probas = top_p_filter(probas, top_p)
        next_token = torch.multinomial(
            probas.cpu(), num_samples=1
        ).to(device)

        token_id = next_token.item()
        generated.append(token_id)

        if (
            tokenizer.eos_token_id is not None
            and token_id == tokenizer.eos_token_id
        ):
            break
        logits = model(next_token, cache=cache)[:, -1]

    full_token_ids = torch.cat(
        [input_ids,
         torch.tensor(generated, device=device, dtype=input_ids.dtype),]
    )
    return full_token_ids, input_ids.numel(), tokenizer.decode(generated)

- Burada yeni bir şey yok
- Yukarıdaki kod, daha önce geliştirdiklerimizin yalnızca daha yalın bir sürümü; 2. bölümdeki [generate_text_basic_stream_cache](https://github.com/rasbt/reasoning-from-scratch/blob/main/reasoning_from_scratch/ch02.py#L57) fonksiyonunu 4. bölümdeki sıcaklık ve top-p örneklemesiyle doğrudan birleştiriyor
- Ayrıca artık her token'ı yield etmek yerine token'ları bir tensörde topluyoruz; çünkü üretilen token'ları canlı olarak yazdırmamız gerekmiyor

In [8]:
torch.manual_seed(0)

raw_prompt = (
    "Half the value of $3x-9$ is $x+37$. "
    "What is the value of $x$?"
)
prompt = render_prompt(raw_prompt)

token_ids, prompt_len, answer_text = sample_response(
            model=model,
            tokenizer=tokenizer,
            prompt=prompt,
            device=device,
            max_new_tokens=512,
            temperature=0.9,
            top_p=0.9,
        )

print(answer_text)

 \boxed{58}<|endoftext|>


In [9]:
torch.manual_seed(5)

token_ids, prompt_len, answer_text = sample_response(
            model=model,
            tokenizer=tokenizer,
            prompt=prompt,
            device=device,
            max_new_tokens=512,
            temperature=0.9,
            top_p=0.9,
        )

print(answer_text)

 Let's solve the problem step by step.

**Given:**
\[
\text{Half the value of } 3x - 9 \text{ is } x + 37.
\]

**Step 1: Translate the statement into an equation.**
\[
\frac{1}{2} (3x - 9) = x + 37
\]

**Step 2: Eliminate the fraction by multiplying both sides by 2.**
\[
3x - 9 = 2(x + 37)
\]

**Step 3: Distribute the 2 on the right side.**
\[
3x - 9 = 2x + 74
\]

**Step 4: Subtract \(2x\) from both sides to get the \(x\)-terms on one side.**
\[
3x - 2x - 9 = 74
\]
\[
x - 9 = 74
\]

**Step 5: Add 9 to both sides to solve for \(x\).**
\[
x = 74 + 9
\]
\[
x = 83
\]

**Final Answer:**
\[
\boxed{83}
\]<|endoftext|>


- Pratikte rollout üretmek için sample_response fonksiyonunu birden çok kez çağırırdık
- GRPO turunu basit tutmak ve şekil 6.13 ile hizalamak için, bunun yerine modelin aşağıdaki dört yanıtı ürettiğini varsayıyoruz:

In [10]:
rollouts = [
    r"\boxed{83}",
    r"The correct answer is \boxed{83}",
    r"The final answer is 83",
    r"We get \boxed{38}",
]

&nbsp;
## 6.6 Ödülleri hesaplamak

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F14_raschka.webp" width=400>

- Ödüller, 3. bölüme benzer şekilde yalnızca doğruluk ödülleridir
- Ancak örtük bir biçim ödülü de vardır: 1.0 ödülü yalnızca nihai yanıt `\boxed{}` biçimindeyse verilir (`fallback=None` aracılığıyla)

In [11]:
from reasoning_from_scratch.ch03 import (
    extract_final_candidate, grade_answer
)

def reward_rlvr(answer_text, ground_truth):
    extracted = extract_final_candidate(
        answer_text, fallback=None  # Require \boxed{}
    )
    if not extracted:
        return 0.0
    correct = grade_answer(extracted, ground_truth)
    return float(correct)

In [12]:
rollouts = [
    r"\boxed{83}",
    r"The correct answer is \boxed{83}",
    r"The final answer is 83",
    r"We get \boxed{38}",
]
rollout_rewards = []

for answer in rollouts:
    reward = reward_rlvr(answer_text=answer, ground_truth="83")
    print(f"Answer: {answer!r}")
    print(f"Reward: {reward}\n")
    rollout_rewards.append(reward)

Answer: '\\boxed{83}'
Reward: 1.0

Answer: 'The correct answer is \\boxed{83}'
Reward: 1.0

Answer: 'The final answer is 83'
Reward: 0.0

Answer: 'We get \\boxed{38}'
Reward: 0.0



- Not: DeepSeek-R1 ekibi, modeli eğitirken ara çözüm adımlarını puanlamak için süreç ödül modelleri kullanmayı denedi
- Ancak bu denemeler başarısız oldu ve araştırmacılar, ara ödüller olmadan yalnızca nihai yanıt doğruluğu ödülleriyle eğitmenin daha iyi olduğu sonucuna vardı

&nbsp;
## 6.7 Rollout'lardan avantajlar aracılığıyla öğrenme sinyalleri hazırlamak

- GRPO'daki "GR" (grup göreli), GRPO'nun istem başına birden çok yanıt (rollout) üretmesine ve bir öğrenme sinyali oluşturmak için bunları birbirine göre karşılaştırmasına gönderme yapar

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F15_raschka.webp" width=400>

- Formül oldukça basit:

$$\text{advantages}_i = \frac{r_i - \mu_r}{\sigma_r + \epsilon}$$

- Burada $r_i$, $i$'inci rollout'un ödülünü; $\mu_r$, rollout grubundaki ortalama ödülü; $\sigma_r$, karşılık gelen standart sapmayı; $\epsilon$ ise sıfıra bölme hatalarından kaçınmak için sayısal kararlılık adına eklenen küçük bir sabiti belirtir

In [13]:
rewards = torch.tensor(rollout_rewards, device=device)
print(rewards)

tensor([1., 1., 0., 0.])


In [14]:
advantages = (rewards - rewards.mean()) / (rewards.std() + 1e-4)

print(advantages)

tensor([ 0.8659,  0.8659, -0.8659, -0.8659])


- Bir gruptaki tüm ödüller aynıysa, örneğin hepsi 0 ya da hepsi 1 ise, tüm $i$ rollout'ları için $r_i - \mu_r = 0$ olduğunu unutmayın
- Bu, tüm yanıtlar doğru ya da tümü yanlışsa modelin güncellenmediği anlamına gelir

&nbsp;
## 6.8 Rollout'ları dizi log-olasılıklarıyla puanlamak

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F16_raschka.webp" width=400>

- Önceki bölümde, yanıt token'ları için token başına log-olasılıkları hesaplayan bir `avg_logprob_answer` fonksiyonu uygulamıştık
- Bu ortalaması alınmış log-olasılıklara çoğu zaman token düzeyinde log-olasılıklar da denir ve LLM yanıtlarını puanlamakta yaygın olarak kullanılır
- Bu ortalama alma, uzunluk normalleştirmesi sağladığı için puanlamada yeğlenir
- Matematiksel olarak bu şöyle ifade edilebilir: $\frac{1}{T} \sum_{t=1}^{T} \log p_W(y_t \mid y_{<t}, x)$
- Burada $y_1, ..., y_T$ üretilen $T$ uzunluğundaki yanıttaki token'ları, $y_{<t}$ daha önce üretilmiş tüm token'ları, $x$ girdi istemini ve $W$ modelin ağırlık parametrelerini belirtir
- Bu ifade, önceki bölümde kullanılanla matematiksel olarak özdeştir; netlik için üretilen çıktı token'larını girdi isteminden ayırt etmek adına yalnızca $x$ yerine $y$ kullanıyoruz
- Başvuru kolaylığı için fonksiyon aşağıya kopyalanmıştır

In [15]:
## Chapter 5

@torch.inference_mode()
def avg_logprob_answer(model, tokenizer, prompt, answer, device="cpu"):

    # İstem uzunluğunu daha sonra elde edebilmek için istem ve yanıt token'larını ayrı ayrı kodla
    prompt_ids = tokenizer.encode(prompt)
    answer_ids = tokenizer.encode(answer)
    full_ids = torch.tensor(prompt_ids + answer_ids, device=device)

    # Daha önceki calc_next_token_logprobas ile aynı
    logits = model(full_ids.unsqueeze(0)).squeeze(0)
    logprobs = torch.log_softmax(logits, dim=-1)

    # Yanıt token'larına karşılık gelen konumların dizin aralığı
    start = len(prompt_ids) - 1
    end = full_ids.shape[0] - 1

    # Öncekiyle aynı, yalnızca start ve end kullanılıyor
    t_idx = torch.arange(start, end, device=device)
    next_tokens = full_ids[start + 1 : end + 1]
    next_token_logps = logprobs[t_idx, next_tokens]

    # Yanıt token puanlarının ortalamasını al
    return torch.mean(next_token_logps).item()

In [16]:
avg_logprob_val = avg_logprob_answer(
                   model, tokenizer, 
                   prompt=prompt,
                   answer=answer_text,
                   device=device) 
print(avg_logprob_val)

-0.061279296875


- Ancak GRPO, yukarıdaki gibi uzunluk normalleştirilmiş token düzeyi ortalamalarını değil, dizi düzeyinde log-olasılıkları kullanır
- Token düzeyi ortalamaları puanlama için faydalıdır; çünkü farklı uzunluklardaki çıktıları karşılaştırılabilir kılar
- GRPO'da her rollout, dizinin tamamı için tek bir ödül ve tek bir avantaj alır; gradyanı doğru ölçeklemek için log-olasılıkların dizinin tamamının olabilirliğini yansıtması gerekir ve bu, token düzeyi log-olasılıkların toplanmasıyla elde edilir
- Aksi hâlde log-olasılıkların ortalamasını almak, öğrenme sinyalini örtük olarak dizi uzunluğuna göre yeniden ölçekler ve özellikle uzun rollout'larda politika güncellemelerini bozar

- Ortalama almayı kaldırıp `torch.mean(next_token_logps)` yerine `torch.sum(next_token_logps)` koyarak bunu dizi düzeyinde bir log-olasılığa dönüştürebiliriz
- Geriye dönük olarak, ortalaması alınmış sonucu yanıt token sayısıyla çarparak da ortalaması alınmamış değeri elde edebiliriz

In [17]:
sequence_logprob_val = avg_logprob_val * (len(tokenizer.encode(answer_text)))
print(sequence_logprob_val)

-16.239013671875


- Bu dizi düzeyi logprob değerleri, T dizi uzunluğuyla doğrusal olarak ölçeklenir
- Bu, daha uzun yanıtların her zaman daha negatif logprob değerleri aldığı anlamına gelir
- Bu da eşit derecede iyi iki yanıt söz konusu olduğunda daha kısa olanın (daha ucuz olanın) yeğlenmesini teşvik eder
- Toplanmış logprob değerleri modeli daha erken durmaya teşvik eder

- Yukarıda belirtildiği gibi, yukarıdaki fonksiyonda `torch.mean` yerine `torch.sum` koyabiliriz
- Ancak önceki bölümde fonksiyonu `@torch.inference_mode()` süsleyicisiyle çıkarım kipinde çalıştırdığımız için, PyTorch'un gradyanları izleyip hesaplamasını istediğimizden onu yine de yeniden tanımlamamız gerekiyor
- Ayrıca 6.5 kısmındaki `sample_response` fonksiyonu `token_ids` ve `prompt_len` değerlerini döndürdüğü için, işleri basitleştirmek adına `avg_logprob_answer` içindeki kodlama ve `full_ids` hesaplamasını kaldırabiliriz

In [18]:
def sequence_logprob_draft(model, token_ids, prompt_len):
    logits = model(token_ids.unsqueeze(0)).squeeze(0).float()
    logprobs = torch.log_softmax(logits, dim=-1)

    # Sonraki token olasılıklarını istediğimiz konumlar
    # Bunlar, t konumundan token_ids[t + 1] değerini tahmin etmeye karşılık gelir
    start = prompt_len - 1
    end = token_ids.shape[0] - 1

    t_idx = torch.arange(start, end, device=token_ids.device)
    next_tokens = token_ids[start + 1 : end + 1]
    next_token_logps = logprobs[t_idx, next_tokens]

    # Yanıt token'ları üzerinde log-olasılıkları topla
    return torch.sum(next_token_logps)

print(sequence_logprob_draft(model, token_ids, prompt_len))

tensor(-16.2998, grad_fn=<SumBackward0>)


- `torch.sum(next_token_logps)` içinde `.item()` kullanmadığımızı, böylece PyTorch'un (Python float yerine) bir tensör döndürdüğünü unutmayın; bu, gradyan hesaplaması için önemlidir
- Görüldüğü gibi, elde edilen değer (-16.2998), `avg_logprob_val` değerini yanıt token sayısıyla yeniden ölçeklediğimizde daha önce elde ettiğimiz değere (-16.2390) neredeyse aynı; küçük farklar kayan nokta yuvarlama davranışına bağlanabilir 

- Aşağıda fonksiyonu, PyTorch'ta biraz daha deyimsel olan ve GPU'lar için biraz daha iyi optimize edilmiş torch.gather kullanarak yeniden yazacağız
- Ancak her iki fonksiyon da matematiksel olarak eşdeğerdir

In [19]:
def sequence_logprob(model, token_ids, prompt_len):
    logits = model(token_ids.unsqueeze(0)).squeeze(0).float()
    logprobs = torch.log_softmax(logits, dim=-1)
    selected = logprobs[:-1].gather(
        1, token_ids[1:].unsqueeze(-1)
    ).squeeze(-1)
    return torch.sum(selected[prompt_len - 1:])

print(sequence_logprob(model, token_ids, prompt_len))

tensor(-16.2998, grad_fn=<SumBackward0>)


In [20]:
rollouts = [
    r"\boxed{83}",
    r"The correct answer is \boxed{83}",
    r"The final answer is 83",
    r"We get \boxed{38}",
]

rollout_logps = []

for text in rollouts:
    token_ids = tokenizer.encode(prompt + " " + text)
    logprob = sequence_logprob(
        model=model,
        token_ids=torch.tensor(token_ids, device=device),
        prompt_len=prompt_len,
    )

    print(f"Answer:  {text}")
    print(f"Logprob: {logprob.item():.4f}\n")

    rollout_logps.append(logprob)

Answer:  \boxed{83}
Logprob: -7.9243

Answer:  The correct answer is \boxed{83}
Logprob: -20.1546

Answer:  The final answer is 83
Logprob: -16.6130

Answer:  We get \boxed{38}
Logprob: -23.3677



- Buradaki eğilim, daha kısa ve daha derli toplu yanıtların daha yüksek (daha az negatif) dizi düzeyi log-olasılıkları almasıdır
- Ve en düşük puan, yanlış bir değer içeren tek yanıta (83 yerine 38) atanmıştır
- Genel olarak, toplanmış log-olasılıklar derli toplu ve doğru çıktıları yeğler

&nbsp;
## 6.9 GRPO kaybı aracılığıyla avantajlardan politika güncellemelerine

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F17_raschka.webp" width=400>

In [21]:
logps = torch.stack(rollout_logps)
print(logps)

tensor([ -7.9243, -20.1546, -16.6130, -23.3677], grad_fn=<StackBackward0>)


In [22]:
pg_loss = -(advantages.detach() * logps).mean()
print(pg_loss)

tensor(-2.5764, grad_fn=<NegBackward0>)


- `.detach()` kullanmamız gerekiyor; çünkü `advantages` değerlerini sabit öğrenme sinyalleri olarak ele almak istiyoruz; böylece yalnızca logprob değerleri üzerinden geri yayılım yaptığımızdan emin oluyoruz
- Negatif işarete ihtiyacımız var; çünkü PyTorch optimize edicileri varsayılan olarak en aza indirir, oysa burada logprob ağırlıklı avantajları en üst düzeye çıkarmak istiyoruz

- Matematiksel gösterimle politika gradyanı kaybını şöyle yazabiliriz:

$$\mathcal{L}_{\mathrm{PG}}
= -\frac{1}{N} \sum_{i=1}^{N} A_i \sum_{t=1}^{T_i} \log p_W\!\left( y_t^{(i)} \mid y_{<t}^{(i)}, x^{(i)} \right)$$

- $N$, yığındaki rollout sayısını belirtir
- $y_1^{(i)}, ..., y_{T_i}^{(i)}$, $T_i$ uzunluğundaki $i$'inci üretilen yanıtın token'larıdır
- $y_{<t}^{(i)}$, o yanıtta daha önce üretilmiş tüm token'ları temsil eder
- $x^{(i)}$, $i$'inci rollout için karşılık gelen girdi istemidir
- $p_W$, modelin politikasını, yani $W$ ağırlıklarıyla parametrelenen sonraki token olasılık dağılımını belirtir
- $A_i$, $i$'inci rollout'un tamamına atanan avantajdır
- İçteki toplam, bir rollout'un dizi düzeyi log-olasılığını hesaplar
- Dıştaki ortalama ise rollout'lar arasında avantaj ağırlıklı log-olasılıkları hesaplar

&nbsp;
## 6.10 Her şeyi bir GRPO adımında bir araya getirmek

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F18_raschka.webp" width=400>

In [23]:
def compute_grpo_loss(
    model,
    tokenizer,
    example,
    device,
    num_rollouts=2,
    max_new_tokens=256,
    temperature=0.8,
    top_p=0.9,
):
    assert num_rollouts >= 2
    roll_logps, roll_rewards, samples = [], [], []
    prompt = render_prompt(example["problem"])

    was_training = model.training
    model.eval()

    for _ in range(num_rollouts):
        # Aşama 1: rollout'ları üret
        token_ids, prompt_len, text = sample_response(
            model=model,
            tokenizer=tokenizer,
            prompt=prompt,
            device=device,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
        )
        # Aşama 2: ödülleri hesapla
        reward = reward_rlvr(text, example["answer"])
        
        # Aşama 4: logprob değerlerini hesapla
        logp = sequence_logprob(model, token_ids, prompt_len)

        roll_logps.append(logp)
        roll_rewards.append(reward)
        samples.append(
            {
                "text": text,
                "reward": reward,
                "gen_len": token_ids.numel() - prompt_len,
            }
        )

    if was_training:
        model.train()

    # Aşama 2: tüm ödülleri topla
    rewards = torch.tensor(roll_rewards, device=device)

    # Aşama 3: avantajları hesapla
    advantages = (rewards - rewards.mean()) / (rewards.std() + 1e-4)

    # Aşama 4: tüm logprob değerlerini topla
    logps = torch.stack(roll_logps)

    # Aşama 5: politika gradyanı kaybını hesapla
    pg_loss = -(advantages.detach() * logps).mean()
    loss = pg_loss  # In the next chapter we add a KL term here

    return {
        "loss": loss.item(),
        "pg_loss": pg_loss.item(),
        "rewards": roll_rewards,
        "advantages": advantages.detach().cpu().tolist(),
        "samples": samples,
        "loss_tensor": loss,
    }

- Kod yorumlarındaki aşamalar, GRPO şeklindeki aşamalarla eşleşir
- 1. aşamadan sonra kod yorumlarında (3 ve 4 yerine) 2 ve 4 aşamalarının geldiğini unutmayın; bu, daha basit bir kod uygulaması sağlar (böylece birden çok for döngüsü yazmak zorunda kalmayız)

In [24]:
torch.manual_seed(123)

stats = compute_grpo_loss(
    model=model,
    tokenizer=tokenizer,
    example=math_train[4],
    device=device,
    num_rollouts=2,
    max_new_tokens=256,
    temperature=0.8,
    top_p=0.9
)

pprint(stats)

{'advantages': [0.0, 0.0],
 'loss': -0.0,
 'loss_tensor': tensor(-0., grad_fn=<NegBackward0>),
 'pg_loss': -0.0,
 'rewards': [0.0, 0.0],
 'samples': [{'gen_len': 4, 'reward': 0.0, 'text': ' 14<|endoftext|>'},
             {'gen_len': 256,
              'reward': 0.0,
              'text': ' 4\n'
                      '\n'
                      "To solve the problem, let's break it down step by "
                      'step:\n'
                      '\n'
                      '1. **Define Variables:**\n'
                      '   - Let \\( x \\) be the number of days Sam works.\n'
                      '   - Then, the number of days he does not work is \\( '
                      '20 - x \\).\n'
                      '\n'
                      '2. **Set Up the Earnings Equation:**\n'
                      '   - For each day he works, he earns \\$60.\n'
                      '   - For each day he does not work, he loses \\$30.\n'
                      '   - His total earnings are \\$660.

&nbsp;
## 6.11 GRPO eğitim döngüsünü uygulamak

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F19_raschka.webp" width=600>

- Zaten maliyetli olan kaynak gereksinimleri nedeniyle yığınlamayı atlıyoruz

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F20_raschka.webp" width=400>

In [25]:
import time

def train_rlvr_grpo(
    model,
    tokenizer,
    math_data,
    device,
    steps=None,
    num_rollouts=2,
    max_new_tokens=256,
    temperature=0.8,
    top_p=0.9,
    lr=1e-5,
    checkpoint_every=50,
    checkpoint_dir=".",
    csv_log_path=None,

):
    if steps is None:
        steps = len(math_data)

    # Aşama 1: optimize ediciyi başlat
    # (model zaten fonksiyonun dışında başlatılmıştı)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    model.train()
    current_step = 0
    if csv_log_path is None:
        timestamp = time.strftime("%Y%m%d_%H%M%S")
        csv_log_path = f"train_rlvr_grpo_metrics_{timestamp}.csv"
    csv_log_path = Path(csv_log_path)

    try:
        # Aşama 2: Eğitim adımları üzerinde döngü kur
        for step in range(steps):

            # Aşama 3: Kayıp gradyanını sıfırla
            # (bunu her adımın başında yapmak iyi bir uygulamadır)
            optimizer.zero_grad()

            current_step = step + 1
            example = math_data[step % len(math_data)]

            # Aşama 4: GRPO kaybını hesapla
            stats = compute_grpo_loss(
                model=model,
                tokenizer=tokenizer,
                example=example,
                device=device,
                num_rollouts=num_rollouts,
                max_new_tokens=max_new_tokens,
                temperature=temperature,
                top_p=top_p,
            )

            # Aşama 5: Kayıp gradyanlarını hesaplamak için geri geçiş
            stats["loss_tensor"].backward()

            # Eğitim kararlılığını artırmak için büyük gradyanları kırp
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            # Aşama 6: Kayıp gradyanlarını kullanarak model ağırlıklarını güncelle
            optimizer.step()

            # Aşama 7: Ödülleri, yanıt uzunluklarını ve kayıpları topla
            reward_avg = torch.tensor(stats["rewards"]).mean().item()
            step_tokens = sum(
                sample["gen_len"] for sample in stats["samples"]
            )
            avg_response_len = (
                step_tokens / len(stats["samples"]) 
                if stats["samples"] else 0.0
            )
            append_csv_metrics(
                csv_log_path, current_step, steps, stats["loss"],
                reward_avg, avg_response_len,
            )

            # Adım ölçütlerini yazdır
            print(
                f"[Step {current_step}/{steps}] "
                f"loss={stats['loss']:.4f} "
                f"reward_avg={reward_avg:.3f} "
                f"avg_resp_len={avg_response_len:.1f}"
            )

            # Modelin tutarlı metin üretip üretmediğini kontrol etmek için
            # örnek çıktılar (her 10 adımda bir)
            if current_step % 10 == 0:
                print(f"[Step {current_step}] sample outputs")
                for i, sample in enumerate(stats["samples"][:3]):
                    text = sample["text"].replace("\n", "\\n")
                    print(
                        f"  {i+1}) reward={sample['reward']:.3f} "
                        f"len={sample['gen_len']}: {text}"
                    )
                print()

            # Aşama 8: Model kontrol noktasını kaydet
            if checkpoint_every and current_step % checkpoint_every == 0:
                ckpt_path = save_checkpoint(
                    model=model,
                    checkpoint_dir=checkpoint_dir,
                    step=current_step,
                )
                print(f"Saved checkpoint to {ckpt_path}")

    # Eğitimi erken keserek durdurursak bir model kontrol noktası kaydet
    except KeyboardInterrupt:
        ckpt_path = save_checkpoint(
            model=model,
            checkpoint_dir=checkpoint_dir,
            step=max(1, current_step),
            suffix="interrupt",
        )
        print(f"\nKeyboardInterrupt. Saved checkpoint to {ckpt_path}")
        return model

    return model


def save_checkpoint(model, checkpoint_dir, step, suffix=""):
    checkpoint_dir = Path(checkpoint_dir)
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    suffix = f"-{suffix}" if suffix else ""
    ckpt_path = (
        checkpoint_dir /
        f"qwen3-0.6B-rlvr-grpo-step{step:05d}{suffix}.pth"
    )
    torch.save(model.state_dict(), ckpt_path)
    return ckpt_path


def append_csv_metrics(
    csv_log_path,
    step_idx,
    total_steps,
    loss,
    reward_avg,
    avg_response_len,
):
    if not csv_log_path.exists():
        csv_log_path.write_text(
            "step,total_steps,loss,reward_avg,avg_response_len\n",
            encoding="utf-8",
        )
    with csv_log_path.open("a", encoding="utf-8") as f:
        f.write(
            f"{step_idx},{total_steps},{loss:.6f},{reward_avg:.6f},"
            f"{avg_response_len:.6f}\n"
        )

- GRPO kayıp hesaplaması olan 4. aşama dışındaki her şey, derin sinir ağları (LLM'ler dahil) eğitilirken kullanılan standart eğitim döngüsünün parçasıdır
- `append_csv_metrics`, kayıt tutmak (ve sonuçları 7. bölümde görselleştirmek) için sonuçları bir CSV dosyasına yazar
- PyTorch'ta sinir ağı eğitimine genel bir giriş için [PyTorch in One Hour: From Tensors to Training Neural Networks on Multiple GPUs](https://sebastianraschka.com/teaching/pytorch-1h/) yazımdaki 3-8. bölümlere bakın

- Kodu 2.7 yerine PyTorch 2.13 ile çalıştırırsanız 5. adımdan itibaren küçük sayısal farklar oluşacağını lütfen unutmayın

In [26]:
device = get_device()
model.to(device)

torch.manual_seed(0)

train_rlvr_grpo(
    model=model,
    tokenizer=tokenizer,
    math_data=math_train,
    device=device,
    steps=50,
    num_rollouts=4,
    max_new_tokens=512,
    temperature=0.8,
    top_p=0.9,
    lr=1e-5,
    checkpoint_every=5,
    checkpoint_dir=".",
    csv_log_path="train_rlvr_grpo_metrics.csv",
)

Using Apple Silicon GPU (MPS)
[Step 1/50] loss=-0.0000 reward_avg=0.000 avg_resp_len=88.0
[Step 2/50] loss=-0.0000 reward_avg=0.000 avg_resp_len=7.5
[Step 3/50] loss=-0.0000 reward_avg=0.000 avg_resp_len=6.5
[Step 4/50] loss=0.0909 reward_avg=0.250 avg_resp_len=6.5
[Step 5/50] loss=1.1001 reward_avg=0.500 avg_resp_len=300.5
Saved checkpoint to qwen3-0.6B-rlvr-grpo-step00005.pth

KeyboardInterrupt. Saved checkpoint to qwen3-0.6B-rlvr-grpo-step00006-interrupt.pth


Qwen3Model(
  (tok_emb): Embedding(151936, 1024)
  (trf_blocks): ModuleList(
    (0-27): 28 x TransformerBlock(
      (att): GroupedQueryAttention(
        (W_query): Linear(in_features=1024, out_features=2048, bias=False)
        (W_key): Linear(in_features=1024, out_features=1024, bias=False)
        (W_value): Linear(in_features=1024, out_features=1024, bias=False)
        (out_proj): Linear(in_features=2048, out_features=1024, bias=False)
        (q_norm): RMSNorm()
        (k_norm): RMSNorm()
      )
      (ff): FeedForward(
        (fc1): Linear(in_features=1024, out_features=3072, bias=False)
        (fc2): Linear(in_features=1024, out_features=3072, bias=False)
        (fc3): Linear(in_features=3072, out_features=1024, bias=False)
      )
      (norm1): RMSNorm()
      (norm2): RMSNorm()
    )
  )
  (final_norm): RMSNorm()
  (out_head): Linear(in_features=1024, out_features=151936, bias=False)
)

- Yukarıdaki kodu çalıştırırken bellekle ilgili sorunlar yaşarsanız rollout sayısını (ör. `num_rollouts=2`) ve rollout başına token sayısını (ör. `max_new_tokens=128`) düşürebilirsiniz
- Ancak görece iyi bir model elde etmek için en az `num_rollouts=8` ve `max_new_tokens=512` gerekir
- Elinizdeki donanımda çalıştıramıyorsanız endişelenmeyin; bir sonraki bölüm önceden eğitilmiş bir kontrol noktasının nasıl indirileceğini gösteriyor

- Her hâlükârda kodun büyük olasılıkla çok yavaş çalışacağını unutmayın; çünkü GRPO kaynak yoğun bir yordamdır
- Çalıştırmayı istediğiniz zaman kesebilirsiniz; en son model kontrol noktasını `checkpoints` klasörüne kaydedecektir
- Bulut GPU'ları kullanmakla ilgileniyorsanız öneriler için [GPU Cloud Resources](../../ch02/02_setup-tips/gpu-instructions.md) belgesine bakın

- Bu kodun yığınlı eğitimi desteklemediğini unutmayın
- Bu, kodu daha basit ve okunabilir tutmak için bilinçli bir tercihtir; ayrıca birden çok (potansiyel olarak uzun) rollout örneklemek zaten çok kaynak yoğun olabilir
- Ancak birden çok GPU'ya erişiminiz varsa, bu kodun yığın ve çoklu GPU destekli isteğe bağlı sürümünü [../02_rlvr_grpo_scripts_intro](../02_rlvr_grpo_scripts_intro) adresindeki ek materyallerde bulabilirsiniz; bu sürüm modeli daha hızlı eğitir

&nbsp;
## 6.12 Kaydedilmiş model kontrol noktalarını yüklemek ve değerlendirmek

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F21_raschka.webp" width=600>

- Kaydedilen kontrol noktaları, 2. bölümde anlatılan model.load_state_dict(torch.load(model_path)) ile yüklenebilir; burada model_path, kontrol noktasının ".pth" dosyasını gösterir
- Bu kontrol noktası dosyaları 3. bölümdeki model değerlendirme araçlarıyla da uyumludur
- Kolaylık olsun diye, 3. bölümün bonus materyallerinde verilen değerlendirme betiklerini kullanabilirsiniz:

```python
uv run ../../ch03/02_math500-verifier-scripts/evaluate_math500.py \
--dataset_size 500 \
--which_model base \
--checkpoint_path checkpoints/qwen3-0.6B-rlvr-grpo-step00050.pth
```
    

- Çok uzun sürdüğü için GRPO eğitimini bilgisayarınızda çalıştırmak istemiyorsanız, [rasbt/qwen3-from-scratch-grpo-checkpoints/tree/main/grpo_original_no_kl](https://huggingface.co/rasbt/qwen3-from-scratch-grpo-checkpoints/tree/main/grpo_original_no_kl) adresine yüklediğim kontrol noktalarını da indirebilirsiniz (indirmek istediğiniz kontrol noktası dosyasına tıklayın ve ardından [download](https://huggingface.co/rasbt/qwen3-from-scratch-grpo-checkpoints/resolve/main/grpo_original_no_kl/qwen3-0.6B-rlvr-grpo-step00050.pth?download=true) düğmesine basın
- Kolaylık olsun diye kontrol noktasını doğrudan burada Python ile de indirebilirsiniz

In [27]:
from reasoning_from_scratch.qwen3 import download_qwen3_grpo_checkpoints

download_qwen3_grpo_checkpoints(grpo_type="no_kl", step="00050")

✓ qwen3-0.6B-rlvr-grpo-step00050.pth already up-to-date


|      | Yöntem                                 | Adım | Maks token | Rollout sayısı | MATH-500 doğr. | Ort. token sayısı |
| ---- | -------------------------------------- | ---- | ---------- | ------------ | ------------ | --------------- |
| 1    | Temel (3. bölüm)                       | -    |            |              | %15.2        | 78.85           |
| 2    | Akıl yürütme (3. bölüm)                | -    |            |              | %48.2        | 1369.79         |
| 3    | KL'siz özgün GRPO (bu bölüm)           | 50   | 512        | 8            | %47.4        | 586.11          | 

- Yukarıdaki tabloya dayanarak, yalnızca 50 adımdan sonra, temel modelden (1. satır) başlatılan eğitilmiş modelin (3. satır) özgün akıl yürütme çeşidi (2. satır) kadar iyi olduğunu görüyoruz
- Daha uzun eğitmenin modeli iyileştirmeyebileceğini, hatta kötüleştirebileceğini unutmayın; çünkü GRPO görece kararsız olabilir; bir sonraki bölüm GRPO algoritmasını iyileştirmek için ek püf noktaları tanıtıyor

&nbsp;
## 6.13 Özet

- Bu bölümde kod yok